# D6 - Advanced Analytics

Bluestock Mutual Fund Analytics Capstone

In [1]:

from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
DB = ROOT / "data" / "db" / "bluestock_mf.db"

con = sqlite3.connect(DB)

funds = pd.read_sql("SELECT * FROM fund_master", con)
nav = pd.read_sql("SELECT * FROM nav_cleaned", con, parse_dates=["date"])
perf = pd.read_sql("SELECT * FROM scheme_performance", con)

print("Funds:", len(funds))
print("NAV rows:", len(nav))
print("Performance rows:", len(perf))


Funds: 40
NAV rows: 64320
Performance rows: 40


## Advanced Risk Analytics

In [2]:

# Historical VaR by fund
var_table = []

for code, group in nav.groupby("amfi_code"):
    r = group.sort_values("date")["nav"].pct_change().dropna()

    var_table.append({
        "amfi_code": code,
        "VaR_95": -np.percentile(r, 5),
        "VaR_99": -np.percentile(r, 1),
        "daily_volatility": r.std()
    })

var_df = pd.DataFrame(var_table).merge(
    funds[["amfi_code","scheme_name","category"]],
    on="amfi_code"
)

var_df["VaR_95_pct"] = var_df["VaR_95"] * 100
var_df["VaR_99_pct"] = var_df["VaR_99"] * 100

display(var_df.sort_values("VaR_95", ascending=False).head(10))


,amfi_code,VaR_95,VaR_99,daily_volatility,scheme_name,category,VaR_95_pct,VaR_99_pct
4,101207,0.023915,0.034725,0.013741,ABSL Small Cap Fund - Regular - Growth,Equity,2.391462,3.472456
17,119095,0.023284,0.033327,0.013351,Axis Small Cap Fund - Regular - Growth,Equity,2.328360,3.332665
22,119599,0.023155,0.033488,0.013289,SBI Small Cap Fund - Direct Plan - Growth,Equity,2.315543,3.348823
11,118634,0.022810,0.033504,0.013447,Nippon India Small Cap Fund - Regular - Growth,Equity,2.280981,3.350434
39,149324,0.021520,0.034205,0.013241,DSP Small Cap Fund - Regular - Growth,Equity,2.152030,3.420481
21,119598,0.021502,0.032123,0.013401,SBI Small Cap Fund - Regular Plan - Growth,Equity,2.150192,3.212281
16,119094,0.016997,0.025741,0.010347,Axis Midcap Fund - Regular - Growth,Equity,1.699654,2.574070
29,120842,0.016950,0.023884,0.009532,Kotak Emerging Equity Fund - Regular - Growth,Equity,1.694994,2.388434
2,100033,0.016902,0.024628,0.010097,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,Equity,1.690163,2.462811
7,102886,0.016857,0.024139,0.009659,UTI Mid Cap Fund - Regular - Growth,Equity,1.685705,2.413877


## Cohort Analysis

In [3]:

tx = pd.read_sql(
    "SELECT * FROM investor_transactions",
    con,
    parse_dates=["transaction_date"]
)

tx["year"] = tx["transaction_date"].dt.year

cohort = (
    tx.groupby(["age_group", "city_tier"])
      .agg(
          transactions=("investor_id","size"),
          investors=("investor_id","nunique"),
          total_amount_inr=("amount_inr","sum"),
          avg_transaction_inr=("amount_inr","mean")
      )
      .reset_index()
      .sort_values("total_amount_inr", ascending=False)
)

display(cohort.head(15))


,age_group,city_tier,transactions,investors,total_amount_inr,avg_transaction_inr
3,26-35,T30,8897,1347,955577822,107404.498370
5,36-45,T30,5558,846,580174250,104385.435408
2,26-35,B30,4566,686,496022396,108633.901883
1,18-25,T30,3044,484,331714080,108973.088042
4,36-45,B30,2588,391,291473278,112624.914219
7,46-55,T30,2600,402,278406605,107079.463462
0,18-25,B30,1872,269,199925312,106797.709402
9,56+,T30,1620,256,173382033,107025.946296
6,46-55,B30,1179,187,126999864,107718.290076
8,56+,B30,854,132,87904790,102933.009368


## Simple Fund Recommender

In [4]:
recommendations = perf.merge(funds[["amfi_code","scheme_name","category","risk_category","expense_ratio_pct"]], on="amfi_code", how="left", suffixes=("_perf",""))
recommendations["expense_ratio_pct"] = recommendations["expense_ratio_pct"].fillna(0)
recommendations["score"] = recommendations["return_3yr_pct"].fillna(0)*0.40 + recommendations["sharpe_ratio"].fillna(0)*20*0.30 - recommendations["expense_ratio_pct"]*0.10 - recommendations["max_drawdown_pct"].abs().fillna(0)*0.20
display(recommendations.sort_values("score", ascending=False)[["scheme_name","category","risk_category","return_3yr_pct","sharpe_ratio","expense_ratio_pct","max_drawdown_pct","score"]].head(10))


,scheme_name,category,risk_category,return_3yr_pct,sharpe_ratio,expense_ratio_pct,max_drawdown_pct,score
14,ICICI Pru Liquid Fund - Regular - Growth,Debt,Low,7.68,7.68,0.74,-2.62,48.554
23,Kotak Liquid Fund - Regular - Growth,Debt,Low,6.18,6.18,0.60,-3.81,38.730
30,ABSL Liquid Fund - Regular - Growth,Debt,Low,5.14,5.14,0.79,-3.66,32.085
9,HDFC Short Term Debt Fund - Regular - Growth,Debt,Low,7.37,1.84,0.56,-6.01,12.730
2,SBI Small Cap Fund - Regular Plan - Growth,Equity,Very High,23.39,0.94,1.43,-13.35,12.183
4,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt,Low,6.07,1.52,0.77,-2.30,11.011
27,Axis Small Cap Fund - Regular - Growth,Equity,Very High,20.98,0.84,1.38,-14.45,10.404
3,SBI Small Cap Fund - Direct Plan - Growth,Equity,Very High,23.14,0.93,0.72,-24.78,9.808
19,Nippon India Gilt Securities Fund - Regular - ...,Debt,Low,5.31,1.33,0.55,-2.23,9.603
29,ABSL Small Cap Fund - Regular - Growth,Equity,Very High,22.38,0.90,1.53,-23.61,9.477



## Conclusion

The advanced analytics layer combines historical VaR, investor cohort analysis and a transparent rule-based recommender. The recommender prioritises medium-term returns and risk-adjusted performance while penalising expenses and drawdown.
